# Library

In [1]:
import os
import time
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

# Mengecek versi TensorFlow dan mengkonfigurasi untuk menggunakan GPU
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU terdeteksi dan dikonfigurasi: {len(gpus)} GPU(s) available.")
    except RuntimeError as e:
        print(e)
else:
    print("GPU tidak terdeteksi. Proses akan menggunakan CPU.")

TensorFlow Version: 2.10.0
GPU terdeteksi dan dikonfigurasi: 1 GPU(s) available.


# Mengecek dan menghapus Gambar rusak dan tidak valid

In [2]:
DATASET_DIR = "dataset_mango"
dataset_path = Path(DATASET_DIR)
invalid_files = []

print("Memulai pemindaian file gambar...")

# Mengecek setiap file di dalam subfolder dataset
for filepath in dataset_path.rglob("*"):
    if filepath.is_file():
        try:
            # Mencoba membuka dan memverifikasi header file gambar
            img = Image.open(filepath)
            img.verify() 
            # Memastikan format file adalah salah satu yang didukung TensorFlow
            if img.format not in ['JPEG', 'PNG', 'GIF', 'BMP']:
                 raise ValueError(f"Format {img.format} tidak didukung TensorFlow")
                 
        except Exception as e:
            print(f"File bermasalah ditemukan: {filepath}")
            print(f"Detail error: {e}")
            invalid_files.append(filepath)

# Tindakan untuk file yang bermasalah
if invalid_files:
    print(f"\nTotal file yg bermasalah: {len(invalid_files)}")
    print("Menghapus file yang bermasalah...")
    
    for file in invalid_files:
        try:
            os.remove(file)
            print(f"File dihapus: {file}")
        except Exception as e:
            print(f"Gagal dihapus {file}: {e}")
else:
    print("Tidak ditemukan file gambar yang bermasalah.")

Memulai pemindaian file gambar...
Tidak ditemukan file gambar yang bermasalah.


# Data augmentation (dipakai atau dijalankan jika data kurang atau bobot tidak seimbang)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

DATASET_DIR = "dataset_mango"

# Inisialisasi generator augmentasi
datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2], # Mengubah kecerahan dari 80% hingga 120%
    fill_mode='nearest'
)

# Jumlah gambar baru yang ingin dibuat per gambar asli
AUG_MULTIPLIER = 2

print("Memulai proses Data Augmentation...")

# Looping ke setiap folder kelas (mango_ripe, mango_rotten, mango_unripe)
for cls in os.listdir(DATASET_DIR):
    class_path = os.path.join(DATASET_DIR, cls)
    
    if os.path.isdir(class_path):
        # Ambil semua file gambar asli dan mengabaikan yg sudah diaugmentasi
        image_files = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg')) and not f.startswith('aug_')]
        
        print(f"Memproses {len(image_files)} gambar asli di kelas: {cls}...")
        
        for img_name in image_files:
            img_path = os.path.join(class_path, img_name)
            try:
                # Muat gambar dan konversi ke array
                img = load_img(img_path)
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape) # Reshape ke format (1, tinggi, lebar, channel)
                
                # Generate dan simpan gambar baru ke folder yang sama
                i = 0
                for batch in datagen.flow(x, batch_size=1, save_to_dir=class_path, save_prefix=f'aug_{img_name.split(".")[0]}', save_format='jpg'):
                    i += 1
                    if i >= AUG_MULTIPLIER:
                        break # Hentikan generator jika sudah mencapai multiplier
            except Exception as e:
                print(f"Gagal memproses gambar {img_name}: {e}")

print("\nData Augmentation Selesai...")

# Loading dataset dan mengkonfigurasi parameter

In [3]:
DATASET_DIR = "dataset_mango"
BATCH_SIZE = 32
IMG_SIZE = (224, 224) 

print("Memuat Training Dataset (80%)...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2, # 20% untuk validation/test, 80% untuk training
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' # Menggunakan one-hot encoding untuk label untuk kompatibilitas dengan MAE
)

print("Memuat sisa dataset untuk Validation & Test (20%)...")
val_test_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' 
)

# Membagi val_test_dataset menjadi Validation (10%) dan Test (10%)
val_batches = tf.data.experimental.cardinality(val_test_dataset) // 2
val_dataset = val_test_dataset.take(val_batches)
test_dataset = val_test_dataset.skip(val_batches)

class_names = train_dataset.class_names
print(f"Kelas yang terdeteksi: {class_names}")

# Menghitung Class Weights untuk menangani ketidakseimbangan kelas
y_train_one_hot = np.concatenate([y for x, y in train_dataset], axis=0)
y_train_sparse = np.argmax(y_train_one_hot, axis=1)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_sparse),
    y=y_train_sparse
)
class_weights = {i: weight for i, weight in enumerate(class_weights_array)}
print(f"Class Weights yang akan digunakan: {class_weights}")

Memuat Training Dataset (80%)...
Found 2625 files belonging to 3 classes.
Using 2100 files for training.
Memuat sisa dataset untuk Validation & Test (20%)...
Found 2625 files belonging to 3 classes.
Using 525 files for validation.
Kelas yang terdeteksi: ['mango_ripe', 'mango_rotten', 'mango_unripe']
Class Weights yang akan digunakan: {0: 1.0086455331412103, 1: 0.9957325746799431, 2: 0.9957325746799431}


# Data preprocessing (Normalisasi ImageNet)

In [4]:
MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def normalize_images(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - MEAN) / STD
    return image, label

# Mengaplikasikan fungsi map dan optimasi memori GPU
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_dataset.map(normalize_images, num_parallel_calls=AUTOTUNE).cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_dataset.map(normalize_images, num_parallel_calls=AUTOTUNE).cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_dataset.map(normalize_images, num_parallel_calls=AUTOTUNE).cache().prefetch(buffer_size=AUTOTUNE)

# Mendefinisikan komponen custom (Dual Attention Mechanism)

In [5]:
# 1. Custom Layer: Channel Attention 
class CustomLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(CustomLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.gamma = self.add_weight(name='gamma', shape=(1, 1, 1, input_shape[-1]), 
                                     initializer='ones', trainable=True)
        super(CustomLayer, self).build(input_shape)

    def call(self, x):
        return x * self.gamma
        
    def get_config(self):
        config = super(CustomLayer, self).get_config()
        return config

# 2. Custom Layer: Spatial Attention 
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)

        self.kernel_size = kernel_size 
        
        self.conv = layers.Conv2D(
            filters=1, 
            kernel_size=kernel_size, 
            padding='same', 
            activation='sigmoid',
            name="spatial_attention_conv"
        )

    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        spatial_mask = self.conv(concat)
        return inputs * spatial_mask

    def get_config(self):
        config = super(SpatialAttention, self).get_config()
        config.update({
            "kernel_size": self.kernel_size,
        })
        return config

# Membuat functional API (Multi-Branch Functional API)

In [6]:
# CELL 5: Model Definition (Functional API + Dual Attention)

def build_freshly_model(input_shape, num_classes):
    # Layer Input
    inputs = layers.Input(shape=input_shape, name="input_image")
    
    # Cabang Warna (Filter 1x1)
    color_branch = layers.Conv2D(32, (1, 1), padding='same', activation='relu', name="color_branch")(inputs)
    color_branch = layers.BatchNormalization()(color_branch)
    
    # Cabang Tekstur (Filter 3x3)
    texture_branch = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name="texture_branch")(inputs)
    texture_branch = layers.BatchNormalization()(texture_branch)
    
    # Cabang Bentuk (Filter 5x5)
    shape_branch = layers.Conv2D(32, (5, 5), padding='same', activation='relu', name="shape_branch")(inputs)
    shape_branch = layers.BatchNormalization()(shape_branch)
    
    # Penggabungan
    combined_features = layers.Concatenate(axis=-1, name="combined_features")([color_branch, texture_branch, shape_branch])
    x = layers.MaxPooling2D((2, 2))(combined_features)
    
    # Deep Feature Extraction
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # 1. Channel Attention: Fokus pada warna/bercak yang penting
    x = CustomLayer(name="texture_color_attention")(x) 
    
    # 2. Spatial Attention: Fokus pada lokasi buah, abaikan daun/pohon
    x = SpatialAttention(name="background_filter_attention")(x)
    
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    # Fully Connected
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output Class
    outputs = layers.Dense(num_classes, activation='softmax', name="output_class")(x)
    
    # Nama model
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="freshly_mango_model")
    return model

# Inisialisasi Model
IMG_SHAPE = IMG_SIZE + (3,) 
NUM_CLASSES = len(class_names)
model_freshly = build_freshly_model(IMG_SHAPE, NUM_CLASSES)
model_freshly.summary()

Model: "freshly_mango_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 color_branch (Conv2D)          (None, 224, 224, 32  128         ['input_image[0][0]']            
                                )                                                                 
                                                                                                  
 texture_branch (Conv2D)        (None, 224, 224, 32  896         ['input_image[0][0]']            
                                )                                               

# Custom training dan evaluation (tf.GradientTape)

In [8]:
# Persiapan Optimizer, Loss, dan Metrics
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
loss_fn = tf.keras.losses.CategoricalCrossentropy()

train_acc_metric = tf.keras.metrics.CategoricalAccuracy()
train_mae_metric = tf.keras.metrics.MeanAbsoluteError()
val_acc_metric = tf.keras.metrics.CategoricalAccuracy()
val_mae_metric = tf.keras.metrics.MeanAbsoluteError()

# Menyiapkan TensorBoard Logging
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = os.path.join('logs', 'mango', 'gradient_tape', current_time, 'train')
val_log_dir = os.path.join('logs', 'mango', 'gradient_tape', current_time, 'val')

# Membuat file log untuk TensorBoard
train_summary_writer = tf.summary.create_file_writer(train_log_dir)
val_summary_writer = tf.summary.create_file_writer(val_log_dir)

# Fungsi Logika Training
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        predictions = model_freshly(x, training=True)
        loss_value = loss_fn(y, predictions)
    grads = tape.gradient(loss_value, model_freshly.trainable_weights)
    optimizer.apply_gradients(zip(grads, model_freshly.trainable_weights))
    
    train_acc_metric.update_state(y, predictions)
    train_mae_metric.update_state(y, predictions)
    return loss_value

# Fungsi Logika Validasi
@tf.function
def val_step(x, y):
    predictions = model_freshly(x, training=False)
    val_loss_value = loss_fn(y, predictions)
    
    val_acc_metric.update_state(y, predictions)
    val_mae_metric.update_state(y, predictions)
    return val_loss_value

# Fase Eksekusi Custom Loop
EPOCHS = 200  
patience = 5
wait = 0
best_val_loss = float('inf')
best_weights = None

print("\nMemulai Training...")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Training Batch Loop
    for step, (x_batch_train, y_batch_train) in enumerate(train_ds):
        train_loss = train_step(x_batch_train, y_batch_train)
        
    train_acc = train_acc_metric.result()
    train_mae = train_mae_metric.result()

    # Mencatat log training ke TensorBoard
    with train_summary_writer.as_default():
        tf.summary.scalar('loss', train_loss, step=epoch)
        tf.summary.scalar('accuracy', train_acc, step=epoch)
        tf.summary.scalar('mae', train_mae, step=epoch)

    # Validation Batch Loop
    val_loss = 0.0
    val_steps = 0
    for x_batch_val, y_batch_val in val_ds:
        val_loss += val_step(x_batch_val, y_batch_val)
        val_steps += 1
    val_loss = val_loss / val_steps 
        
    val_acc = val_acc_metric.result()
    val_mae = val_mae_metric.result()

    # Mencatat log validasi ke TensorBoard
    with val_summary_writer.as_default():
        tf.summary.scalar('loss', val_loss, step=epoch)
        tf.summary.scalar('accuracy', val_acc, step=epoch)
        tf.summary.scalar('mae', val_mae, step=epoch)
    
    waktu = time.time() - start_time
    print(f"Epoch {epoch+1}/{EPOCHS} ({waktu:.0f}s) \ntrain_loss: {train_loss:.2f} - train_acc: {train_acc:.2f} - train_mae: {train_mae:.2f} \nval_loss: {val_loss:.2f} - val_acc: {val_acc:.2f} - val_mae: {val_mae:.2f}")

    # Early Stopping dengan Kriteria Ganda (Accuracy > 85% dan MAE < 0.02)
    if val_acc > 0.85 and val_mae < 0.02:
        print(f"Target Tercapai, Menyimpan model...")
        if best_weights is not None:
            model_freshly.set_weights(best_weights)
        model_freshly.save('freshly_model_mango.h5')
        break
        
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        best_weights = model_freshly.get_weights()
    else:
        wait += 1
        if wait >= patience:
            print(f"Batas patience tercapai, menyimpan model terbaik...")
            if best_weights is not None:
                model_freshly.set_weights(best_weights)
            model_freshly.save('freshly_model_mango.h5')
            break

    # Reset metrics
    train_acc_metric.reset_states()
    train_mae_metric.reset_states()
    val_acc_metric.reset_states()
    val_mae_metric.reset_states()


Memulai Training...
Epoch 1/200 (27s) 
train_loss: 0.29 - train_acc: 0.78 - train_mae: 0.21 
val_loss: 0.80 - val_acc: 0.76 - val_mae: 0.36
Epoch 2/200 (15s) 
train_loss: 0.26 - train_acc: 0.87 - train_mae: 0.12 
val_loss: 0.79 - val_acc: 0.51 - val_mae: 0.31
Epoch 3/200 (15s) 
train_loss: 0.15 - train_acc: 0.90 - train_mae: 0.10 
val_loss: 0.68 - val_acc: 0.71 - val_mae: 0.23
Epoch 4/200 (15s) 
train_loss: 0.13 - train_acc: 0.91 - train_mae: 0.09 
val_loss: 0.38 - val_acc: 0.82 - val_mae: 0.13
Epoch 5/200 (15s) 
train_loss: 0.35 - train_acc: 0.92 - train_mae: 0.08 
val_loss: 0.49 - val_acc: 0.79 - val_mae: 0.14
Epoch 6/200 (15s) 
train_loss: 0.12 - train_acc: 0.94 - train_mae: 0.06 
val_loss: 0.25 - val_acc: 0.91 - val_mae: 0.09
Epoch 7/200 (15s) 
train_loss: 0.20 - train_acc: 0.94 - train_mae: 0.06 
val_loss: 0.23 - val_acc: 0.91 - val_mae: 0.07
Epoch 8/200 (15s) 
train_loss: 0.28 - train_acc: 0.96 - train_mae: 0.05 
val_loss: 0.18 - val_acc: 0.94 - val_mae: 0.06
Epoch 9/200 (15s) 


# Evaluasi tes

In [9]:
print("\nPersiapan Evaluasi pada Test Dataset...")

# Kompilasi ulang model dengan metrics yang diperlukan untuk evaluasi akhir
model_freshly.compile(
    loss='categorical_crossentropy',
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.MeanAbsoluteError(name='mae')
    ]
)
# Evaluasi pada Test Dataset
test_loss, test_acc, test_mae = model_freshly.evaluate(test_ds)

print("\n" + "="*30)
print("HASIL EVALUASI AKHIR")
print("="*30)
print(f"Test Loss     : {test_loss:.2f}")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test MAE      : {test_mae:.2f}")
print("="*30)



Persiapan Evaluasi pada Test Dataset...
9/9 [==============================] - 2s 128ms/step - loss: 0.1193 - accuracy: 0.9591 - mae: 0.0344

HASIL EVALUASI AKHIR
Test Loss     : 0.12
Test Accuracy : 95.91%
Test MAE      : 0.03


# Fine Tuning jika target akurasi tes belum mencapai 85% atau MAE diatas 0.02

In [10]:
print("Memuat model terbaik sebelumnya...")
# Memuat model dengan akurasi terbaik
model_freshly_tuning = tf.keras.models.load_model(
    'freshly_model_mango.h5', 
    compile=False,
    custom_objects={
        'CustomLayer': CustomLayer,
        'SpatialAttention': SpatialAttention
    }
)

# Gunakan Learning Rate yang SANGAT KECIL (0.00001)
optimizer_sharp = tf.keras.optimizers.Adam(learning_rate=1e-5)
loss_fn_sharp = tf.keras.losses.CategoricalCrossentropy() # Pastikan TANPA label_smoothing

# Siapkan Metrik
acc_metric = tf.keras.metrics.CategoricalAccuracy()
mae_metric = tf.keras.metrics.MeanAbsoluteError()

@tf.function
def sharpen_step(x, y):
    with tf.GradientTape() as tape:
        predictions = model_freshly_tuning(x, training=True)
        # Gunakan loss yang sama dengan training awal, tapi tanpa label smoothing untuk fokus pada perbaikan prediksi akhir
        loss = loss_fn_sharp(y, predictions)
    grads = tape.gradient(loss, model_freshly_tuning.trainable_weights)
    optimizer_sharp.apply_gradients(zip(grads, model_freshly_tuning.trainable_weights))
    return loss, predictions

print("\nMemulai proses Fine Tuning...")
TUNING_EPOCHS = 30

for epoch in range(TUNING_EPOCHS):
    start_time = time.time()
    
    # Training Loop
    for x_batch, y_batch in train_ds:
        loss_val, preds = sharpen_step(x_batch, y_batch)
        acc_metric.update_state(y_batch, preds)
        mae_metric.update_state(y_batch, preds)
        
    train_acc = acc_metric.result()
    train_mae = mae_metric.result()
    
    # Evaluation Loop
    val_acc_metric.reset_states()
    val_mae_metric.reset_states()
    
    for x_val, y_val in val_ds:
        val_preds = model_freshly_tuning(x_val, training=False)
        val_acc_metric.update_state(y_val, val_preds)
        val_mae_metric.update_state(y_val, val_preds)
        
    val_acc = val_acc_metric.result()
    val_mae = val_mae_metric.result()
    
    waktu = time.time() - start_time
    print(f"Epoch {epoch+1}/{TUNING_EPOCHS} ({waktu:.0f}s) - acc: {train_acc:.2f} - mae: {train_mae:.2f} - val_acc: {val_acc:.2f} - val_mae: {val_mae:.2f}")
    
    # Jika MAE Validation sudah di bawah 0.015 langsung berhenti dan simpan model
    if val_mae <= 0.015:
        print(f"MAE sudah mencapai {val_mae:.4f}. Menyimpan model hasil tuning...")
        model_freshly_tuning.save('freshly_model_mango_tuning.h5')
        break

    acc_metric.reset_states()
    mae_metric.reset_states()

Memuat model terbaik sebelumnya...

Memulai proses Fine Tuning...
Epoch 1/30 (17s) - acc: 0.98 - mae: 0.02 - val_acc: 0.96 - val_mae: 0.03
Epoch 2/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.96 - val_mae: 0.03
Epoch 3/30 (15s) - acc: 0.98 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 4/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.96 - val_mae: 0.02
Epoch 5/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.96 - val_mae: 0.02
Epoch 6/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 7/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 8/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 9/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 10/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 11/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 12/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.97 - val_mae: 0.02
Epoch 13/30 (15s) - acc: 0.99 - mae: 0.02 - val_acc: 0.

In [26]:
model_freshly_tuning.save('freshly_model_mango_tuning.h5')

# Evaluasi tes setelah fine tuning

In [11]:
print("\nPersiapan Evaluasi pada Test hasil Fine Tuning...")

# Kompilasi ulang model dengan metrics yang diperlukan untuk evaluasi akhir
model_freshly_tuning.compile(
    loss='categorical_crossentropy',
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.MeanAbsoluteError(name='mae')
    ]
)
# Evaluasi pada Test Dataset
test_loss, test_acc, test_mae = model_freshly_tuning.evaluate(test_ds)

print("\n" + "="*30)
print("HASIL EVALUASI AKHIR")
print("="*30)
print(f"Test Loss     : {test_loss:.2f}")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test MAE      : {test_mae:.2f}")
print("="*30)


Persiapan Evaluasi pada Test hasil Fine Tuning...
9/9 [==============================] - 1s 55ms/step - loss: 0.0967 - accuracy: 0.9740 - mae: 0.0203

HASIL EVALUASI AKHIR
Test Loss     : 0.10
Test Accuracy : 97.40%
Test MAE      : 0.02


# Simpan ke SavedModel

In [25]:
# Menyimpan dalam format direktori TensorFlow SavedModel 
model_freshly_tuning.save("mango_saved_model")
print("\nModel disimpan ke folder 'mango_saved_model'...")

INFO:tensorflow:Assets written to: mango_saved_model\assets


INFO:tensorflow:Assets written to: mango_saved_model\assets



Model disimpan ke folder 'mango_saved_model'...
